In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
from datasets import load_dataset
dataset = load_dataset("SetFit/sst5")

In [ ]:
import pandas as pd

df1 = dataset["train"].to_pandas()

df2 = dataset["test"].to_pandas()

df3 = dataset["validation"].to_pandas()

df = pd.concat([df1, df2, df3])
df.head()

In [ ]:
def create_balanced_dataset(df):

    # Count the instances of 0
    num_spam = df[df["label"] == 0].shape[0]

    # Randomly sample other instances to match the number of 0 instances
    label1_subset = df[df["label"] == 1].sample(num_spam, random_state=123)
    label2_subset = df[df["label"] == 2].sample(num_spam, random_state=123)
    label3_subset = df[df["label"] == 3].sample(num_spam, random_state=123)
    label4_subset = df[df["label"] == 4].sample(num_spam, random_state=123)

    # Combine other subsets with 0
    balanced_df = pd.concat([label1_subset, label2_subset, label3_subset, label4_subset, df[df["label"] == 0]])

    return balanced_df

balanced_df = create_balanced_dataset(df)
balanced_df = balanced_df.drop("label_text", axis=1)
print(balanced_df["label"].value_counts())

In [ ]:
def random_split(df, train_frac, validation_frac):
    # Shuffle the entire DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Calculate split indices
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# Test size is implied to be 0.2 as the remainder

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [ ]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()
print(f"Major: {major_version}, Minor: {minor_version}")
from trl import SFTTrainer
import numpy as np
import os
import pandas as pd
import numpy as np
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, Trainer
from typing import Tuple
import warnings
from typing import Any, Dict, List, Union
from transformers import DataCollatorForLanguageModeling
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from unsloth import FastModel

NUM_CLASSES = 5 # number of classes in the dataset

max_seq_length = 2048 # Choose any
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

model_name = "unsloth/gemma-4-E2B";load_in_4bit = False

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,load_in_4bit = load_in_4bit,
    max_seq_length = max_seq_length,
    dtype = dtype,
)

In [ ]:
from peft import LoftQConfig

model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
import datasets

prompt_template = """### Instruction:
Classify the sentiment of this movie review into one of the classes (0, 1, 2, 3, 4):
- 0: Very Negative
- 1: Negative
- 2: Neutral
- 3: Positive
- 4: Very Positive

Review:
{text}

### Response:
{label}"""

# apply prompt on texts
train_df['formatted_text'] = [
    prompt_template.format(text=t.strip(), label=int(l)) + tokenizer.eos_token
    for t, l in zip(train_df['text'], train_df['label'])
]

validation_df['formatted_text'] = [
    prompt_template.format(text=t.strip(), label=int(l)) + tokenizer.eos_token
    for t, l in zip(validation_df['text'], validation_df['label'])
]

# create new datasets
train_dataset = datasets.Dataset.from_pandas(train_df[['formatted_text']], preserve_index=False)
validation_dataset = datasets.Dataset.from_pandas(validation_df[['formatted_text']], preserve_index=False)


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only




# create trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    dataset_text_field = "formatted_text",
    max_seq_length = 512,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 180,
        learning_rate = 2e-4,
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 30,
        save_steps = 30,                     
        save_total_limit = 2,                
        load_best_model_at_end = True,       
        metric_for_best_model = "eval_loss", 
        greater_is_better = False,
        per_device_eval_batch_size = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),

)

# apply mask on responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "### Instruction:\n",
    response_part = "### Response:\n",
)

In [ ]:
# start training
trainer_stats = trainer.train()

In [ ]:
import re
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_prompt_template = """### Instruction:
Classify the sentiment of this movie review into one of the classes (0, 1, 2, 3, 4):
- 0: Very Negative
- 1: Negative
- 2: Neutral
- 3: Positive
- 4: Very Positive

Review:
{text}

### Response:
"""

predictions = []
ground_truths = []
unmatched_count = 0
sample = test_df[:250]
print(f"Starting evaluation on {len(sample)} test samples...")

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Testing"):
    review_text = str(row["text"]).strip()
    true_label = int(row["label"])

    prompt = test_prompt_template.format(text=review_text)
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 2,
            use_cache = True,
            temperature = 0.01  # Greedy
        )

    prompt_len = inputs.input_ids.shape[1]
    pred_text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()

    match = re.search(r"[0-4]", pred_text)
    if match:
        pred_label = int(match.group(0))
    else:
        pred_label = -1
        unmatched_count += 1

    predictions.append(pred_label)
    ground_truths.append(true_label)

acc = accuracy_score(ground_truths, predictions) * 100
print(f"\n==========================================")
print(f"Final Test Accuracy: {acc:.2f}%")
print(f"Unmatched Predictions: {unmatched_count}")
print(f"==========================================\n")

target_names = ["0: Very Neg", "1: Neg", "2: Neutral", "3: Pos", "4: Very Pos"]
print(classification_report(ground_truths, predictions, labels=[0, 1, 2, 3, 4], target_names=target_names))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Visualize confusion matrix for Gemma4-e2b Classifier
sns.heatmap(confusion_matrix(ground_truths,predictions),annot= True, cmap = 'coolwarm', fmt='.0f')
plt.title('confusion matrix')
plt.ylabel('Predicted Values')
plt.xlabel('Actual Values')
plt.show()

In [ ]:
model.save_pretrained("gemma_classification_lora")  # Local saving
tokenizer.save_pretrained("gemma_classification_lora")